> **Nota de contexto (léela antes de empezar):** este notebook usa "embeddings" (vectores que representan texto) para poder aplicarles PCA y visualizarlos. Todavía no hemos explicado formalmente qué es un embedding ni cómo se construye — eso lo veremos varias sesiones después, en `02 Temas selectos/03 Aplicaciones NLP (word embeddings)`. Por ahora basta con la idea intuitiva: cada fila de texto del dataset ya viene representada como un vector de números (la columna `embedding`), y aquí solo aplicamos la misma técnica de reducción de dimensionalidad (PCA) que acabas de ver con datos numéricos normales, para poder graficar esos vectores en 3D. Cuando llegues a la sección de embeddings, vale la pena regresar a este notebook con esa base ya construida.


# Bibliotecas
---


In [ ]:
# Dependencies
import re
import os
import sys
import io
import pandas as pd
import zipfile
import json
import numpy as np
import ast
from sklearn.decomposition import PCA
import plotly.express as px
#
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Dataset Original:
# https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/data?select=Combined+Data.csv

# Usaremos solo el 5% del dataset, la selección fue leatoria con:
# df_combined_sample = df_combined.sample(frac=0.05, random_state=42)

In [ ]:
# Los datos se leen directamente desde el zip, sin extraer un CSV plano a disco
# (el zip pesa 36MB; el CSV sin comprimir pesa 92MB).
zip_file_path = 'Data_embedding.csv.zip'
csv_file_name = 'Data_embedding.csv'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    with zip_ref.open(csv_file_name) as f:
        df = pd.read_csv(f)

df.head()

In [ ]:
# Display the frequency table of the 'status' column
status_counts = df['status'].value_counts()
display(status_counts)

In [ ]:
# --- 1) Preparar X (matriz de embeddings) ---
cols_req = ["embedding", "status", "statement"]
mask = df[cols_req].notna().all(axis=1)
df_ = df.loc[mask, cols_req].copy()


In [ ]:
# Convierte la columna embedding (lista/array por fila en formato string) en una matriz 2D (n_muestras x dim)
# Usa ast.literal_eval para convertir las strings de lista en listas de Python
X = np.array([ast.literal_eval(e) for e in df_["embedding"].tolist()], dtype=float)

#assert X.ndim == 2, "Cada 'embedding' debe ser una lista/array de igual longitud."
#assert np.isfinite(X).all(), "Hay valores no finitos (NaN/Inf) en los embeddings."

In [ ]:
# --- 2) PCA a 3 componentes ---
# Nota: PCA centra automáticamente las features (no escala a var=1).
pca = PCA(n_components=3, random_state=0)
X_pca = pca.fit_transform(X)  # shape: (n_muestras, 3)


In [ ]:
# --- 3) DataFrame para graficar ---
df_plot = pd.DataFrame(
    X_pca, columns=["PC1", "PC2", "PC3"], index=df_.index
).assign(
    status=df_["status"].astype(str),
    statement=df_["statement"].astype(str)
)

In [ ]:
# Títulos con varianza explicada
var = pca.explained_variance_ratio_
axis_titles = {
    "x": f"PC1 ({var[0]:.1%})",
    "y": f"PC2 ({var[1]:.1%})",
    "z": f"PC3 ({var[2]:.1%})",
}
axis_titles

In [ ]:
# --- 4) Gráfica 3D interactiva con Plotly ---
fig = px.scatter_3d(
    df_plot, x="PC1", y="PC2", z="PC3",
    color="status",
    hover_data={"statement": True, "status": True, "PC1": ':.3f', "PC2": ':.3f', "PC3": ':.3f'},
    opacity=0.85,
    width=900, height=650,
    title="PCA de embeddings (3D)"
)

fig.update_layout(
    scene=dict(
        xaxis_title=axis_titles["x"],
        yaxis_title=axis_titles["y"],
        zaxis_title=axis_titles["z"],
    ),
    legend_title_text="status",
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

## Para pensar

**Nota de contexto:** este notebook usa "embeddings" (vectores que representan texto) antes de que el concepto se explique formalmente — eso ocurre hasta la Sesión 10 (`02 Temas selectos/03 Aplicaciones NLP`). Por ahora basta con la idea intuitiva: cada fila de texto ya viene representada como un vector de números, y aquí solo se le aplica PCA a esos vectores para poder visualizarlos en 3D.

1. En la gráfica 3D, ¿los puntos con el mismo valor de `status` tienden a agruparse, o están mezclados con los de otros `status`? ¿Qué te dice eso sobre si el embedding original captura información relevante para esa variable?

2. Revisa el porcentaje de varianza explicada por las 3 componentes usadas para graficar (aparece en los títulos de los ejes). ¿Es un porcentaje alto o bajo respecto al total? Si es bajo, ¿qué implica eso sobre confiar demasiado en lo que "se ve" en la gráfica 3D?

3. **Ejercicio:** cuando llegues a la Sesión 10 y veas cómo se genera un embedding desde cero (`03 Aplicaciones NLP/04 Intro_to_Embeddings`), regresa a este notebook y pregúntate: ¿de qué texto o proceso crees que salió la columna `embedding` que se usa aquí?
